In [6]:
import pandas as pd
import openpyxl

In [7]:
df_with_features = pd.read_excel("1_with_features.xlsx")
df_with_info_city = pd.read_excel("Города.xlsx")
df_with_geo_data = pd.read_excel("Парсинг.xlsx")

In [8]:
df_with_features.head()


,Название,Цена,URL объявления,Описание,Дата публикации,Продавец,Город,Адрес,Img,URL группы,...,near_water,has_concierge,has_storage_room,has_panoramic_windows,has_balcony,has_loggia,has_terrace,has_mortgage,is_assignment,has_discount
0,"2-к. квартира, 48,1 м², 17/17 эт.",5964400,https://www.avito.ru//kirovskaya_oblast_kirov/...,"ЖК «Скандинавия», ул. Анжелия Михеева 5.\n\nВ ...",2025-09-22 13:37:50,i198476227,Киров,"ул. Анжелия Михеева, д. 5",https://50.img.avito.st/image/1/1.8_XckLa6Xxyq...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,False,False,False,False,False,False,False,False,False,False
1,"1-к. квартира, 39,1 м², 11/12 эт.",5829810,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:46:23,i275700606,Киров,"ул. Свободы, д. 141",https://60.img.avito.st/image/1/1.VMI9qba6-CtL...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,False,False,True,True,True,False,False,False,False,False
2,"3-к. квартира, 52,9 м², 17/17 эт.",6400900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Сохраняем традиции и создаём новые. Слобода — ...,2025-09-29 10:26:27,i198476227,Киров,"ул. Потребкооперации, д. 34, корп. 1",https://50.img.avito.st/image/1/1.8QtqMLa6XeIc...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,False,False,False,False,False,False,False,False,False,False
3,"1-к. квартира, 39 м², 10/12 эт.",5814900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:37:16,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.6FsUTra6RLJi...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,False,False,True,True,True,False,False,False,False,False
4,"1-к. квартира, 40,1 м², 12/12 эт.",6315750,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:42:24,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.BinIDra6qsC-...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,False,False,True,True,True,False,False,False,False,False


In [9]:
df_with_info_city.head()

,№,Unnamed: 1,Субъект РФ,Население,Сгенерированная ссылка Avito,Индекс города,Курортный,"Cредняя зп в городе, тыс руб (2025)",2015 население,Динамика населения за 10 лет,"ВРП района 2023, млн руб"
0,Категория: 300 000 - 499 999 человек,NaN,NaN,NaN,NaN,https://индекс-городов.рф/#/,NaN,NaN,NaN,NaN,NaN
1,1,Киров,Кировская область,475871.0,https://www.avito.ru/kirovskaya_oblast_kirov/k...,233,N,69.2,493336.0,-17465.0,605915.5
2,2,Чебоксары,Чувашия (Чувашская республика),497061.0,https://www.avito.ru/cheboksary/kvartiry/proda...,243,N,67.5,473895.0,23166.0,601316.7
3,3,Улан-Удэ,Бурятия,435067.0,https://www.avito.ru/ulan-ude/kvartiry/prodam/...,197,N,73.2,426650.0,8417.0,503918.8
4,4,Калининград,Калининградская область,488690.0,https://www.avito.ru/kaliningrad/kvartiry/prod...,270,N,76.5,453461.0,35229.0,785205.9


In [10]:

_distance_cols = [c for c in df_with_geo_data.columns if c.startswith('dist_')]
_geo_subset_cols = ['URL объявления'] + _distance_cols
_geo_subset_cols = [c for c in _geo_subset_cols if c in df_with_geo_data.columns]
geo_subset = df_with_geo_data[_geo_subset_cols].drop_duplicates(subset=['URL объявления'])


cities = df_with_info_city.rename(columns={'Unnamed: 1': 'Город'}).copy()
cities['Город'] = cities['Город'].astype(str).str.strip()

if '№' in cities.columns:
    cities = cities[~cities['№'].astype(str).str.contains('Категория', na=False)]
cities = cities[cities['Город'].notna() & (cities['Город'] != 'nan')]

_city_cols_preferred = [
    'Город',
    'Субъект РФ',
    'Население',
    'Индекс города',
    'Курортный',
    'Cредняя зп в городе, тыс руб (2025)',
    '2015 население',
    'Динамика населения за 10 лет',
    'ВРП района 2023, млн руб',
]
_city_cols = [c for c in _city_cols_preferred if c in cities.columns]
cities_subset = cities[_city_cols].drop_duplicates(subset=['Город'])


merged = df_with_features.merge(geo_subset, on='URL объявления', how='left')

merged = merged.merge(cities_subset, on='Город', how='left')

print('Merged shape:', merged.shape)
print('Distance columns added:', _distance_cols)


output_path = 'dataset_merged.xlsx'
merged.to_excel(output_path, index=False)

print('Saved to:', output_path)

merged.head()


Merged shape: (18822, 57)
Distance columns added: ['dist_to_city_center', 'dist_to_school', 'dist_to_kindergarten', 'dist_to_park', 'dist_to_bus_stop', 'dist_to_supermarket']
Saved to: dataset_merged.xlsx


,Название,Цена,URL объявления,Описание,Дата публикации,Продавец,Город,Адрес,Img,URL группы,...,dist_to_park,dist_to_bus_stop,dist_to_supermarket,Субъект РФ,Индекс города,Курортный,"Cредняя зп в городе, тыс руб (2025)",2015 население,Динамика населения за 10 лет,"ВРП района 2023, млн руб"
0,"2-к. квартира, 48,1 м², 17/17 эт.",5964400,https://www.avito.ru//kirovskaya_oblast_kirov/...,"ЖК «Скандинавия», ул. Анжелия Михеева 5.\n\nВ ...",2025-09-22 13:37:50,i198476227,Киров,"ул. Анжелия Михеева, д. 5",https://50.img.avito.st/image/1/1.8_XckLa6Xxyq...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.41,0.25,0.06,Кировская область,233,N,69.2,493336.0,-17465.0,605915.5
1,"1-к. квартира, 39,1 м², 11/12 эт.",5829810,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:46:23,i275700606,Киров,"ул. Свободы, д. 141",https://60.img.avito.st/image/1/1.VMI9qba6-CtL...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233,N,69.2,493336.0,-17465.0,605915.5
2,"3-к. квартира, 52,9 м², 17/17 эт.",6400900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Сохраняем традиции и создаём новые. Слобода — ...,2025-09-29 10:26:27,i198476227,Киров,"ул. Потребкооперации, д. 34, корп. 1",https://50.img.avito.st/image/1/1.8QtqMLa6XeIc...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,1.59,0.45,0.53,Кировская область,233,N,69.2,493336.0,-17465.0,605915.5
3,"1-к. квартира, 39 м², 10/12 эт.",5814900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:37:16,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.6FsUTra6RLJi...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233,N,69.2,493336.0,-17465.0,605915.5
4,"1-к. квартира, 40,1 м², 12/12 эт.",6315750,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:42:24,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.BinIDra6qsC-...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233,N,69.2,493336.0,-17465.0,605915.5
